# Import modules

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Import Multi-task ElasticNet

In [ ]:
import joblib

model_1 = joblib.load('ElasticNet.pkl')

In [ ]:
model_features = model_1.feature_names_in_
model_features

In [ ]:
import pickle

with open('y_train_features.pkl', 'rb') as f:
    y_train_features_loaded = pickle.load(f)

print(y_train_features_loaded)

# Import Random forest regressor

In [ ]:
model_2 = joblib.load('Random Forest Regressor.pkl')

# Import ensemble

In [ ]:
final_estimator = joblib.load('Ensemble.pkl')

# RNA-Seq data

In [ ]:
counts_mcf7 = pd.read_csv('RawCounts.csv')
counts_mcf7 = counts_mcf7.rename(columns = {'Gene_ID' : 'gene_id'})

counts_mcf7 = counts_mcf7[['gene_id', 'eV_DM1', 'eV_DM2', 'eV_DM3',
                           'PI3K_WT_DM1', 'PI3K_WT_DM2', 'PI3K_WT_DM3']]

counts_mcf7 = counts_mcf7.rename(columns = {'eV_DM1' : 'MCF7_E545K_1',
                                            'eV_DM2' : 'MCF7_E545K_2',
                                            'eV_DM3' : 'MCF7_E545K_3',
                                            'PI3K_WT_DM1' : 'MCF7_WT_1',
                                            'PI3K_WT_DM2' : 'MCF7_WT_2',
                                            'PI3K_WT_DM3' : 'MCF7_WT_3'})

counts_mcf7 = counts_mcf7.set_index('gene_id')
counts_mcf7

In [ ]:
counts_mcf10a = pd.read_csv('counts_matrix_filtered_MCF_PI3K.csv')
counts_mcf10a = counts_mcf10a.rename(columns = {'Unnamed: 0' : 'gene_id'})


# #compare by excluding E5_3

counts_mcf10a = counts_mcf10a[['gene_id', 'MCF_WT_1', 'MCF_WT_2', 'MCF_WT_3',
                               'MCF_E5_1', 'MCF_E5_2',
                               'MCF_H10_1', 'MCF_H10_2', 'MCF_H10_3']]


counts_mcf10a = counts_mcf10a.rename(columns = {'MCF_WT_1' : 'MCF10A_WT_1', 'MCF_WT_2' : 'MCF10A_WT_2', 'MCF_WT_3' : 'MCF10A_WT_3',
                                                'MCF_E5_1' : 'MCF10A_E545K_1', 'MCF_E5_2' : 'MCF10A_E545K_2',
                                                'MCF_H10_1' : 'MCF10A_H1047R_1', 'MCF_H10_2' : 'MCF10A_H1047R_2', 'MCF_H10_3' : 'MCF10A_H1047R_3'})
counts_mcf10a = counts_mcf10a.set_index('gene_id')
counts_mcf10a

# TPM normalization

In [ ]:
#TMM normalization with rnanorm
!pip install rnanorm
from rnanorm import TPM

In [ ]:
def tpm_normalize(counts):
  counts = counts.T
  norm_counts = TPM(gtf = 'Homo_sapiens.GRCh38.113.gtf').fit(counts)
  norm_counts = norm_counts.transform(counts)
  norm_counts = pd.DataFrame(norm_counts)
  norm_counts.columns = counts.columns
  norm_counts.index = counts.index

  return norm_counts

In [ ]:
mcf7_tpm = tpm_normalize(counts_mcf7)
mcf10a_tpm = tpm_normalize(counts_mcf10a)

In [ ]:
mcf7_tpm.head(3)

In [ ]:
mcf10a_tpm.head(3)

# Select genes in metabolism

In [ ]:
met_genes = pd.read_csv('human_gem_associated_genes.csv')
met_genes

In [ ]:
def select_met_genes(df):
  met_genes_select = met_genes[met_genes['gene_id'].isin(df.columns)]
  met_genes_select = met_genes_select.gene_id.to_list()
  print('N of metabolism related genes: ' + str(len(met_genes_select)))
  df = df[met_genes_select]
  df = df.T
  df = pd.concat([met_genes.set_index('gene_id'), df], axis = 'columns')
  df = df.set_index('Symbol')
  df.index.name = None
  df = df.T

  return df

In [ ]:
mcf7_met = select_met_genes(mcf7_tpm)
mcf10a_met = select_met_genes(mcf10a_tpm)

In [ ]:
mcf7_met.head(3)

In [ ]:
mcf10a_met.head(3)

# Metabolite prediction

In [ ]:
def filter_model_features(df):
  filtered_data = df[model_features]
  return filtered_data

In [ ]:
mcf7_met_filtered = filter_model_features(mcf7_met)
mcf10a_met_filtered = filter_model_features(mcf10a_met)

In [ ]:
mcf7_met_filtered.head(2)

In [ ]:
mcf10a_met_filtered.head(2)

In [ ]:
def predict_metabolites(model, filtered_data):
  pred_metabolites = model.predict(filtered_data)
  pred_metabolites = pd.DataFrame(pred_metabolites, index = filtered_data.index, columns = y_train_features_loaded)
  return pred_metabolites

In [ ]:
#Elastic Net predictions
mcf7_pred_elastic = predict_metabolites(model_1, mcf7_met_filtered)
mcf10a_pred_elastic = predict_metabolites(model_1, mcf10a_met_filtered)

#Random forest regressor predictions
mcf7_pred_rf = predict_metabolites(model_2, mcf7_met_filtered)
mcf10a_pred_rf = predict_metabolites(model_2, mcf10a_met_filtered)

In [ ]:
#Ensemble predictions

def predict_metabolites_ensemble(model_1, model_2, final_estimator, filtered_data):
  pred_1 = model_1.predict(filtered_data)
  pred_2 = model_2.predict(filtered_data)
  stacked_preds = np.hstack([pred_1, pred_2])
  pred_metabolites = final_estimator.predict(stacked_preds)
  pred_metabolites = pd.DataFrame(pred_metabolites, index = filtered_data.index, columns = y_train_features_loaded)

  return pred_metabolites

mcf7_pred_ensemble = predict_metabolites_ensemble(model_1, model_2, final_estimator, mcf7_met_filtered)
mcf10a_pred_ensemble = predict_metabolites_ensemble(model_1, model_2, final_estimator, mcf10a_met_filtered)

# LCMS data

In [ ]:
lcms = pd.read_excel('MCF10A_MCF7_LCMS_normalized.xlsx', index_col = 0)
lcms

In [ ]:
mcf7_lcms = lcms.iloc[0:10, :]
mcf7_lcms

In [ ]:
mcf7_lcms.drop(columns = ['acetylcholine', 'lactate', 'kynurenic acid'], inplace = True)
mcf7_lcms

In [ ]:
mcf10a_lcms = lcms.iloc[10:, :]
mcf10a_lcms

In [ ]:
mcf10a_lcms.drop(columns = ['acetylcholine', 'lactate', 'kynurenic acid', 'allantoin', 'cystathionine', 'glycodeoxycholate/glycochenodeoxycholate',
                            'serotonin', 'guanosine'], inplace = True)
mcf10a_lcms

In [ ]:
from sklearn.impute import SimpleImputer

def filter_and_impute_missing_values(df, set_threshold):
  threshold = set_threshold
  nan_percentage = df.isnull().mean()
  filtered_df = df.loc[:, nan_percentage <= threshold]

  imputer = SimpleImputer(missing_values = np.nan, strategy = 'median')
  res = imputer.fit_transform(filtered_df)
  res = pd.DataFrame(res, columns = filtered_df.columns)
  res.index = filtered_df.index

  return res

In [ ]:
mcf7_lcms_clean_imputed = filter_and_impute_missing_values(df = mcf7_lcms, set_threshold = 0.2)
mcf10a_lcms_clean_imputed = filter_and_impute_missing_values(df = mcf10a_lcms, set_threshold = 0.2)

In [ ]:
mcf7_lcms_clean_imputed

In [ ]:
mcf10a_lcms_clean_imputed

# Comparison of predictions vs actuals

In [ ]:
def filter_common_metabolites(predictions_df, lcms_df):
  common_metabolites = list(set(predictions_df.columns).intersection(set(lcms_df.columns)))
  predictions_df = predictions_df[common_metabolites]
  lcms_df = lcms_df[common_metabolites]
  return predictions_df, lcms_df

In [ ]:
mcf7_pred_elastic_filtered, mcf7_lcms_filtered = filter_common_metabolites(mcf7_pred_elastic, mcf7_lcms_clean_imputed)
mcf7_pred_rf_filtered, mcf7_lcms_filtered = filter_common_metabolites(mcf7_pred_rf, mcf7_lcms_clean_imputed)
mcf7_pred_ensemble_filtered, mcf7_lcms_filtered = filter_common_metabolites(mcf7_pred_ensemble, mcf7_lcms_clean_imputed)

mcf10a_pred_elastic_filtered, mcf10a_lcms_filtered = filter_common_metabolites(mcf10a_pred_elastic, mcf10a_lcms_clean_imputed)
mcf10a_pred_rf_filtered, mcf10a_lcms_filtered = filter_common_metabolites(mcf10a_pred_rf, mcf10a_lcms_clean_imputed)
mcf10a_pred_ensemble_filtered, mcf10a_lcms_filtered = filter_common_metabolites(mcf10a_pred_ensemble, mcf10a_lcms_clean_imputed)

In [ ]:
mcf7_pred_elastic_filtered['PI3K_status'] = ['MUT', 'MUT', 'MUT','WT', 'WT', 'WT']
mcf7_pred_rf_filtered['PI3K_status'] = ['MUT', 'MUT', 'MUT','WT', 'WT', 'WT']
mcf7_pred_ensemble_filtered['PI3K_status'] = ['MUT', 'MUT', 'MUT','WT', 'WT', 'WT']

In [ ]:
mcf7_lcms_filtered['PI3K_status'] = ['WT', 'WT', 'WT', 'WT', 'WT',
                                     'MUT', 'MUT', 'MUT','MUT', 'MUT']
mcf7_lcms_filtered

In [ ]:
mcf10a_pred_elastic_filtered['PI3K_status'] = ['WT', 'WT', 'WT', 'MUT', 'MUT', 'MUT','MUT', 'MUT']
mcf10a_pred_rf_filtered['PI3K_status'] = ['WT', 'WT', 'WT', 'MUT', 'MUT', 'MUT','MUT', 'MUT']
mcf10a_pred_ensemble_filtered['PI3K_status'] = ['WT', 'WT', 'WT', 'MUT', 'MUT', 'MUT','MUT', 'MUT']

In [ ]:
mcf10a_lcms_filtered['PI3K_status'] = ['WT', 'WT', 'WT', 'WT', 'WT',
                                       'MUT', 'MUT', 'MUT','MUT', 'MUT',
                                       'MUT', 'MUT', 'MUT','MUT', 'MUT']
mcf10a_lcms_filtered

In [ ]:
def get_mean_and_direction_of_change_by_group(df):
  res = df.groupby('PI3K_status').mean()
  res = res.transpose()
  res['Higher_in_MUT'] = res['MUT'] > res['WT']
  res.replace({True : 'yes', False : 'no'}, inplace = True)
  return res

In [ ]:
mcf7_pred_elastic_filtered_mean = get_mean_and_direction_of_change_by_group(mcf7_pred_elastic_filtered)
mcf7_pred_rf_filtered_mean = get_mean_and_direction_of_change_by_group(mcf7_pred_rf_filtered)
mcf7_pred_ensemble_filtered_mean = get_mean_and_direction_of_change_by_group(mcf7_pred_ensemble_filtered)
mcf7_lcms_filtered_mean = get_mean_and_direction_of_change_by_group(mcf7_lcms_filtered)

mcf10a_pred_elastic_filtered_mean = get_mean_and_direction_of_change_by_group(mcf10a_pred_elastic_filtered)
mcf10a_pred_rf_filtered_mean = get_mean_and_direction_of_change_by_group(mcf10a_pred_rf_filtered)
mcf10a_pred_ensemble_filtered_mean = get_mean_and_direction_of_change_by_group(mcf10a_pred_ensemble_filtered)
mcf10a_lcms_filtered_mean = get_mean_and_direction_of_change_by_group(mcf10a_lcms_filtered)

In [ ]:
def get_pred_vs_actual_comparison(df_elastic, df_rf, df_ensemble, df_lcms, cell_line_name):
  #Elastic Net
  df1 = pd.concat([df_elastic.rename(columns = {'Higher_in_MUT' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_MUT' : 'actual'})['actual']], axis = 'columns')
  df1['comparison'] = df1['prediction'] == df1['actual']
  print('Correctly predicted metabolites from Elastic Net: ' + str(list(df1[df1['comparison'] == True].index)))
  res1 = pd.DataFrame(df1['comparison'].value_counts()).reindex([True, False])
  res1.rename(columns = {'count' : 'ElasticNet'}, inplace = True)

  #Random forest regressor
  df2 = pd.concat([df_rf.rename(columns = {'Higher_in_MUT' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_MUT' : 'actual'})['actual']], axis = 'columns')
  df2['comparison'] = df2['prediction'] == df2['actual']
  print('Correctly predicted metabolites from Random forest regressor: ' + str(list(df2[df2['comparison'] == True].index)))
  res2 = pd.DataFrame(df2['comparison'].value_counts()).reindex([True, False])
  res2.rename(columns = {'count' : 'RF regressor'}, inplace = True)

  #Ensemble
  df3 = pd.concat([df_ensemble.rename(columns = {'Higher_in_MUT' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_MUT' : 'actual'})['actual']], axis = 'columns')
  df3['comparison'] = df3['prediction'] == df3['actual']
  print('Correctly predicted metabolites from Ensemble: ' + str(list(df3[df3['comparison'] == True].index)))
  res3 = pd.DataFrame(df3['comparison'].value_counts()).reindex([True, False])
  res3.rename(columns = {'count' : 'Ensemble'}, inplace = True)

  #Final result
  res = pd.concat([res1, res2, res3], axis = 'columns')

  res_transposed = res.T
  res_transposed.plot(kind='bar', stacked=True)
  locs, labels = plt.yticks()

  new_labels = [int(float(label.get_text())) for label in labels]
  plt.yticks(locs, new_labels)

  plt.title(f'Comparison of Predictions vs. Actuals ({cell_line_name})')
  plt.ylabel('Count')
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
  plt.savefig(f'{cell_line_name}_pred_vs_actual.svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
#MCF7
get_pred_vs_actual_comparison(df_elastic = mcf7_pred_elastic_filtered_mean,
                              df_rf = mcf7_pred_rf_filtered_mean,
                              df_ensemble = mcf7_pred_ensemble_filtered_mean,
                              df_lcms = mcf7_lcms_filtered_mean,
                              cell_line_name = 'MCF7')

In [ ]:
#MCF10A
get_pred_vs_actual_comparison(df_elastic = mcf10a_pred_elastic_filtered_mean,
                              df_rf = mcf10a_pred_rf_filtered_mean,
                              df_ensemble = mcf10a_pred_ensemble_filtered_mean,
                              df_lcms = mcf10a_lcms_filtered_mean,
                              cell_line_name = 'MCF10A')

In [ ]:
mcf7_correct_metabolites = ['malate', 'glutamine', 'allantoin', 'creatinine', 'alanine', 'uridine', 'hypoxanthine', 'adenine', 'NAD', 'threonine', 'proline', 'asparagine', 'xanthine', 'methionine', 'homocysteine', 'aspartate', 'leucine', 'glycine', 'glycodeoxycholate/glycochenodeoxycholate', 'serine', 'glutathione reduced', 'citrulline', 'histidine', 'ornithine', 'GABA', 'phosphocreatine']

mcf10a_correct_metabolites = ['creatine', 'malate', 'glutamine', 'alanine', 'fumarate/maleate/alpha-ketoisovalerate', 'AMP', 'hypoxanthine', 'phenylalanine', 'threonine', 'asparagine', 'homocysteine', 'glutamate', 'UMP', 'glycine', 'arginine', 'citrulline', 'phosphocreatine']

In [ ]:
!pip install matplotlib-venn

In [ ]:
from matplotlib_venn import venn2

In [ ]:
venn2([ set(mcf7_correct_metabolites), set(mcf10a_correct_metabolites)], set_labels=('MCF7', 'MCF10A'))
plt.title('Comparison of correctly predicted metabolites in MCF7 and MCF10A models')
plt.savefig('MCF10A_MCF7_common_correct_metabolites_venn.svg', bbox_inches = 'tight')
plt.show()

In [ ]:
correct_common = list(set(mcf7_correct_metabolites).intersection(set(mcf10a_correct_metabolites)))
correct_common

In [ ]:
with open('MCF10A_MCF7_correct_common_metabolites.txt', 'w') as f:
    for item in correct_common:
      f.write("%s\n" % item)

# LCMS statistics and further plots

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
def get_statistically_significant_metabolites(df, df_mean, pval_threshold):
  grouped = df.groupby('PI3K_status')
  mut = grouped.get_group('MUT').drop(columns=['PI3K_status'])
  wt = grouped.get_group('WT').drop(columns=['PI3K_status'])
  results = {}
  for col in mut.columns:
    t_stat, p_value = ttest_ind(mut[col], wt[col])
    results[col] = {'T-statistic': t_stat, 'P-value': p_value}
  results_df = pd.DataFrame(results).T
  results_df.index = mut.columns
  df_mean = df_mean.copy()
  df_mean.index.name = None
  res = pd.concat([df_mean, results_df], axis = 'columns')
  return res

In [ ]:
mcf7_sig = get_statistically_significant_metabolites(df = mcf7_lcms_filtered, df_mean = mcf7_lcms_filtered_mean,  pval_threshold = 0.05)
mcf7_sig.head(2)

In [ ]:
mcf10a_sig = get_statistically_significant_metabolites(df = mcf10a_lcms_filtered, df_mean = mcf10a_lcms_filtered_mean,  pval_threshold = 0.05)
mcf10a_sig.head(2)

In [ ]:
def get_log2fc(df):
  df_res = df
  df_res['log2FC'] = np.log2(df_res['WT'] / df_res['MUT'])
  return   df_res

In [ ]:
mf10a_log2fc = get_log2fc(mcf10a_sig)
mf7_log2fc = get_log2fc(mcf7_sig)

In [ ]:
mf10a_log2fc_sorted = mf10a_log2fc.sort_values(by=['log2FC'], ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=mf10a_log2fc_sorted, x=mf10a_log2fc_sorted.index, y='log2FC',
            hue='P-value', palette='viridis', dodge=False)

# Customize the plot
plt.title('Bar Plot of Metabolite log2FC (MCF10A WT vs MUT)')
plt.xlabel('Metabolite')
plt.ylabel('log2 Fold Change')
plt.xticks(rotation=90)
plt.legend(title='P-value', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
mf7_log2fc_sorted = mf7_log2fc.sort_values(by=['log2FC'], ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=mf7_log2fc_sorted, x=mf7_log2fc_sorted.index, y='log2FC',
            hue='P-value', palette='viridis', dodge=False)

plt.title('Bar Plot of Metabolite log2FC (MCF7 WT vs MUT)')
plt.xlabel('Metabolite')
plt.ylabel('log2 Fold Change')
plt.xticks(rotation=90)
plt.legend(title='P-value', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
combined = pd.concat([mf7_log2fc_sorted.rename(columns = {'P-value' : 'mcf7_pval', 'log2FC' : 'mcf7_log2FC'})[['mcf7_pval', 'mcf7_log2FC']],
                      mf10a_log2fc_sorted.rename(columns = {'P-value' : 'mcf10a_pval', 'log2FC' : 'mcf10a_log2FC'})[['mcf10a_pval', 'mcf10a_log2FC']]],
                      axis = 'columns')
combined

In [ ]:
combined = combined.dropna()
combined = combined[(combined['mcf7_pval'] < 0.05) & (combined['mcf10a_pval'] < 0.05)]
combined

In [ ]:
combined_heatmap_data = combined.rename(columns = {'mcf7_log2FC': 'MCF7', 'mcf10a_log2FC': 'MCF10A'})[['MCF7', 'MCF10A']]
combined_heatmap_data

In [ ]:
plt.figure(figsize=(5, 10))
plt.title('log2FC (WT vs MUT) of altered metabolites (LCMS, p-value < 0.05)')
sns.heatmap(data = combined_heatmap_data, linewidths = 0.01, cmap = 'viridis')

In [ ]:
combination = list(set(correct_common).intersection(set(list(combined_heatmap_data.index))))
(len(combination) * 100/ len(correct_common))

In [ ]:
set1 = set(correct_common)
set2 = set(list(combined_heatmap_data.index))

venn2([set1, set2], set_labels=('Correctly Predicted', 'Statistically Significant (LCMS, common in both cell lines)'))
plt.title('Overlap Between Correct Predictions and Statistically Significant Metabolites')
plt.show()

In [ ]:
#MCF7
get_pred_vs_actual_comparison(df_elastic = mcf7_pred_elastic_filtered_mean.loc[combination],
                              df_rf = mcf7_pred_rf_filtered_mean.loc[combination],
                              df_ensemble = mcf7_pred_ensemble_filtered_mean.loc[combination],
                              df_lcms = mcf7_lcms_filtered_mean.loc[combination],
                              cell_line_name = 'MCF7_stat_sig')

In [ ]:
#MCF10A
get_pred_vs_actual_comparison(df_elastic = mcf10a_pred_elastic_filtered_mean.loc[combination],
                              df_rf = mcf10a_pred_rf_filtered_mean.loc[combination],
                              df_ensemble = mcf10a_pred_ensemble_filtered_mean.loc[combination],
                              df_lcms = mcf10a_lcms_filtered_mean.loc[combination],
                              cell_line_name = 'MCF10A_stat_sig')